In [2]:
import os
from dataclasses import dataclass
from typing import Any, Dict, List
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.tools import tool, ToolRuntime

from dotenv import load_dotenv
load_dotenv()

model = ChatOpenAI(model="gpt-4.1-mini")

In [3]:
# Real-world Example: Customer Support System with Context
# This demonstrates how ToolRuntime provides context to tools

USER_DATABASE = {
    "user123": {
        "name": "Alice Johnson", 
        "account_type": "Premium",
        "balance": 5000,
        "email": "alice@example.com",
        "support_tier": "Priority"
    },
    "user456": {
        "name": "Bob Smith",
        "account_type": "Standard", 
        "balance": 1200,
        "email": "bob@example.com",
        "support_tier": "Standard"
    }
}

@dataclass
class UserContext:
    user_id: str
    session_id: str = "default"

@tool
def get_account_info(runtime: ToolRuntime[UserContext]) -> str:
    """Get the current user's account information."""
    user_id = runtime.context.user_id
    
    if user_id in USER_DATABASE:
        user = USER_DATABASE[user_id]
        return f"""Account Information:
        Name: {user['name']}
        Type: {user['account_type']}  
        Balance: ${user['balance']}
        Email: {user['email']}
        Support Tier: {user['support_tier']}"""
    return "User not found in system"

@tool
def update_support_tier(new_tier: str, runtime: ToolRuntime[UserContext]) -> str:
    """Update user's support tier (Priority, Standard, Basic)."""
    user_id = runtime.context.user_id
    
    if user_id in USER_DATABASE:
        USER_DATABASE[user_id]['support_tier'] = new_tier
        return f"Support tier updated to {new_tier} for user {USER_DATABASE[user_id]['name']}"
    return "User not found"

# Create agent with context
agent = create_agent(
    model,
    tools=[get_account_info, update_support_tier],
    context_schema=UserContext,
    system_prompt="You are a customer support assistant. Help users with their account inquiries."
)

# Test with real agent - Context is automatically injected into tools
print("=== Testing Context Access ===")
result = agent.invoke(
    {"messages": [{"role": "user", "content": "What's my current account information?"}]},
    context=UserContext(user_id="user123", session_id="session_001")
)

=== Testing Context Access ===


In [4]:
print("\nAgent Response:")
print(result["messages"][-1].content)


Agent Response:
Your current account information is as follows:
- Name: Alice Johnson
- Account Type: Premium
- Balance: $5000
- Email: alice@example.com
- Support Tier: Priority

Is there anything specific you would like to do with your account?


In [5]:
# Namespace structure (tuple of segments)
namespace = ("category", "subcategory", "user_id")

# Examples:
("preferences", "user_prefs")
("notes", "personal")
("tasks", "user123", "workspace_alpha")
("app", "memory", "user123")   # per-user LTM bucket (used in this lab)

('app', 'memory', 'user123')

In [6]:
@dataclass
class UserContext:
    user_id: str


def memory_namespace(user_id: str) -> tuple[str, ...]:
    """One LTM bucket per user — use this in every save/retrieve tool."""
    return ("app", "memory", user_id)

In [7]:
import os
from dataclasses import dataclass
from typing import List

from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.tools import tool, ToolRuntime

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore

from dotenv import load_dotenv
load_dotenv()

model = ChatOpenAI(model="gpt-4.1-mini")


@dataclass
class UserContext:
    user_id: str


def memory_namespace(user_id: str) -> tuple[str, ...]:
    return ("app", "memory", user_id)

In [8]:
LTM_KEY = "messages"


@tool
def save_ltm(data: str, runtime: ToolRuntime[UserContext]) -> str:
    """Save a line of information to this user's long-term memory."""
    store = runtime.store
    ns = memory_namespace(runtime.context.user_id)

    item = store.get(ns, LTM_KEY)
    if item is None:
        messages: List[str] = []
    else:
        messages = list(item.value) if isinstance(item.value, list) else [str(item.value)]

    messages.append(data)
    store.put(ns, LTM_KEY, messages)
    print(f"Saving to LTM for {runtime.context.user_id}:", data)
    return "Data saved to long-term memory."


@tool
def retrieve_ltm(data: str, runtime: ToolRuntime[UserContext]) -> str:
    """Retrieve this user's long-term memory (data arg can hint what to look for)."""
    store = runtime.store
    ns = memory_namespace(runtime.context.user_id)

    print(f"Retrieving from LTM for {runtime.context.user_id}")
    item = store.get(ns, LTM_KEY)
    if item is None:
        return "No data in long-term memory."
    return str(item.value)

In [9]:
agent = create_agent(
    tools=[save_ltm, retrieve_ltm],
    model=model,
    store=InMemoryStore(),
    checkpointer=InMemorySaver(),
    context_schema=UserContext,
    system_prompt="""You are a helpful assistant that can answer any queries.
If the user shares personal information, preferences, or explicit approvals,
save that information using the save_ltm tool.
Before answering a query when prior facts might matter, retrieve long-term
memory using retrieve_ltm and use that context in your answer.
""",
)

In [10]:
config_siva_t1 = {"configurable": {"thread_id": "siva-chat-1"}}
siva = UserContext(user_id="siva")

response_1 = agent.invoke(
    {"messages": [{"role": "user", "content": "My name is Siva"}]},
    config=config_siva_t1,
    context=siva,
)
response_1

Saving to LTM for siva: User's name is Siva


{'messages': [HumanMessage(content='My name is Siva', additional_kwargs={}, response_metadata={}, id='30aaf8a2-9ebe-4462-a5be-a6860d9fac2f'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 147, 'total_tokens': 167, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_162d0a701a', 'id': 'chatcmpl-EH3L9tGexjQHcZHBiaGsv0fY1wm7X', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a03d32-a00c-7a62-9e3c-e3a126df28ff-0', tool_calls=[{'name': 'save_ltm', 'args': {'data': "User's name is Siva"}, 'id': 'call_ccuUzoP225Yf2bTN7lVDrxfU

In [11]:
config_siva_t2 = {"configurable": {"thread_id": "siva-chat-2"}}

response_2 = agent.invoke(
    {"messages": [{"role": "user", "content": "What is my name?"}]},
    config=config_siva_t2,
    context=siva,
)
response_2

Retrieving from LTM for siva


{'messages': [HumanMessage(content='What is my name?', additional_kwargs={}, response_metadata={}, id='53cc5875-a92d-4d2e-9167-871b37665f6b'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 147, 'total_tokens': 162, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_162d0a701a', 'id': 'chatcmpl-EH3LBsJCl2Jl0zWI5Igj60azcjwX6', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a03d32-a858-7183-b13e-3837c46c4981-0', tool_calls=[{'name': 'retrieve_ltm', 'args': {'data': 'name'}, 'id': 'call_qgoHv0jqCcmtmUvXDrWSdm0s', 'type':

In [12]:
config_alex_t1 = {"configurable": {"thread_id": "alex-chat-1"}}
alex = UserContext(user_id="alex")

response_3 = agent.invoke(
    {"messages": [{"role": "user", "content": "What is my name?"}]},
    config=config_alex_t1,
    context=alex,
)
response_3

Retrieving from LTM for alex


{'messages': [HumanMessage(content='What is my name?', additional_kwargs={}, response_metadata={}, id='e5e5b2c3-13c9-4e3d-9a5e-bfa0f9a8d9c9'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 147, 'total_tokens': 162, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_162d0a701a', 'id': 'chatcmpl-EH3LCQvZsKwlZwmvXSMgjXWioMiUA', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a03d32-afed-7302-b90f-5aeba05923b4-0', tool_calls=[{'name': 'retrieve_ltm', 'args': {'data': 'name'}, 'id': 'call_hCkRGt19sAQlJMrFJBZ36uaD', 'type':